In [2]:
import sagemaker
from sagemaker.estimator import Estimator
import boto3

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
region = boto3.Session().region_name
role = sagemaker.get_execution_role()
session = sagemaker.Session()
bucket = session.default_bucket()

image_uri = "864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-byoc:latest"

print("region:", region)
print("role:", role)
print("bucket:", bucket)
print("image_uri:", image_uri)

region: us-east-1
role: arn:aws:iam::864475311845:role/SageMakerStudioExecutionRole2026
bucket: sagemaker-us-east-1-864475311845
image_uri: 864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-byoc:latest


In [4]:
import os
print(os.listdir("data/prep"))

['.gitkeep', 'features_train.csv.gz']


In [5]:
train_input = session.upload_data(
    path="data/prep/features_train.csv.gz",
    bucket=bucket,
    key_prefix="future-sales/train"
)

print(train_input)

s3://sagemaker-us-east-1-864475311845/future-sales/train/features_train.csv.gz


In [6]:
estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/future-sales/output",
    sagemaker_session=session
)

In [7]:
estimator.fit(
    {
        "train": train_input
    }
)

INFO:sagemaker:Creating training-job with name: future-sales-byoc-2026-03-08-00-57-25-181


2026-03-08 00:57:26 Starting - Starting the training job...
2026-03-08 00:57:41 Starting - Preparing the instances for training...
2026-03-08 00:58:03 Downloading - Downloading input data...
2026-03-08 00:58:38 Training - Training image download completed. Training in progress.2026-03-08 00:58:42,562 - train - INFO - Logger initialized. log_path=artifacts/logs/train_20260308_005842.log
2026-03-08 00:58:42,563 - train - INFO - Starting train step
2026-03-08 00:58:42,563 - train - INFO - HAS_LGB=True
2026-03-08 00:58:42,564 - train - INFO - prepared_features_path=/opt/program/data/prep/features_train.csv.gz
2026-03-08 00:58:42,564 - train - INFO - model_output_path=/opt/program/artifacts/models/final_model.joblib
2026-03-08 00:58:42,564 - train - INFO - val_block=33
2026-03-08 00:58:48,037 - train - INFO - Loaded features rows=10913850 cols=8
2026-03-08 00:58:48,038 - train - INFO - n_feature_columns=7
2026-03-08 00:58:49,357 - train - INFO - train_df rows=10675678
2026-03-08 00:58:49,35

In [8]:
print(estimator.model_data)

s3://sagemaker-us-east-1-864475311845/future-sales/output/future-sales-byoc-2026-03-08-00-57-25-181/output/model.tar.gz


In [9]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import pandas as pd

In [10]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

INFO:sagemaker:Creating model with name: future-sales-byoc-2026-03-08-01-05-44-484
INFO:sagemaker:Creating endpoint-config with name future-sales-byoc-2026-03-08-01-05-44-484
INFO:sagemaker:Creating endpoint with name future-sales-byoc-2026-03-08-01-05-44-484


-----!

In [11]:
print(predictor.endpoint_name)

future-sales-byoc-2026-03-08-01-05-44-484


In [ ]:
###

In [3]:
import sagemaker
import boto3

session = sagemaker.Session()
region = boto3.Session().region_name

print(region)

us-east-1


In [4]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

endpoint_name = "future-sales-byoc-2026-03-08-01-05-44-484"

predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

In [5]:
payload = {
    "instances": [
        {
            "date_block_num": 33,
            "shop_id": 59,
            "item_id": 5037,
            "item_category_id": 19,
            "item_cnt_month_lag_1": 1.0,
            "item_cnt_month_lag_2": 0.0,
            "item_cnt_month_lag_3": 0.0,
        }
    ]
}

result = predictor.predict(payload)
print(result)

{'predictions': [0.48299161640458155]}
